# CHASEDB1 Distance Map from GT

This notebook generates distance maps from CHASEDB1 ground-truth vessel masks stored in `gt/`.
It writes outputs to `gt_dist/` as `.npy` (float distance) and optional `.png` previews.

In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image
from scipy.ndimage import distance_transform_edt
import matplotlib.pyplot as plt
import autorootcwd
import torch
from types import SimpleNamespace

CUDA_VISIBLE_DEVICES = "1"
# Set default CUDA device to cuda:1
torch.cuda.set_device(1)
DEVICE = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Global Device Set: {DEVICE}")
print(f"🔧 Default CUDA Device: cuda:{torch.cuda.current_device()}")

In [ ]:
data_dir = Path('data/chase')
gt_dir = Path(data_dir, 'gt')
out_dir = Path(data_dir, 'gt_dist')
out_inverted_dir = Path(data_dir, 'gt_dist_inverted')
out_dir.mkdir(parents=True, exist_ok=True)
out_inverted_dir.mkdir(parents=True, exist_ok=True)

gt_files = sorted(gt_dir.glob('*.png'))
print(f'Found {len(gt_files)} GT masks')
print(gt_files[:3])

In [ ]:
# Inspect a sample mask
sample_path = gt_files[0]
mask = np.array(Image.open(sample_path).convert('L'))
print(sample_path.name, mask.shape, mask.dtype)
print('Unique values:', np.unique(mask))

plt.imshow(mask, cmap='gray')
plt.title(sample_path.name)
plt.axis('off')
plt.show()

In [ ]:
modes = ['signed', 'inverted']
save_preview_png = True  # Set to True to save visual previews of distance maps

# Aggregate mean of pre-normalized dist_map values across all samples
raw_dist_sum = 0.0
raw_dist_count = 0
raw_dist_sum_by_mode = {m: 0.0 for m in modes}
raw_dist_count_by_mode = {m: 0 for m in modes}

for p in gt_files:
    for mode in modes:
        mask = np.array(Image.open(p).convert('L'))
        dist = distance_transform_edt(mask)

        # Save raw distance as .npy
        if mode == 'signed':
            out_npy = out_dir / (p.stem + f'_dist_{mode}.npy')
        else:
            out_npy = out_inverted_dir / (p.stem + f'_dist_{mode}.npy')
        if mode == 'inverted':
            dist[dist > 0] = dist.max() - dist[dist > 0]  # Invert sign for outside distances

        # Track pre-normalization statistics
        raw_dist_sum += float(dist.sum())
        raw_dist_count += dist.size
        raw_dist_sum_by_mode[mode] += float(dist.sum())
        raw_dist_count_by_mode[mode] += dist.size

        dist = (dist - dist.min()) / (dist.max() - dist.min())
        # print(dist.min(), dist.max())
        np.save(out_npy, dist)

        if save_preview_png:
            # Normalize for visualization
            vmax = np.percentile(dist, 99) if mode == 'outside' else np.percentile(np.abs(dist), 99)
            norm = np.clip(dist, -vmax, vmax)
            if mode == 'outside':
                vis = (norm / vmax * 255.0).astype(np.uint8)
                cmap = 'magma'
            else:
                vis = ((norm + vmax) / (2 * vmax) * 255.0).astype(np.uint8)
                cmap = 'seismic'
            if mode == 'signed':
                out_png = out_dir / (p.stem + f'_dist_{mode}.png')
            else:
                out_png = out_inverted_dir / (p.stem + f'_dist_{mode}.png')
            Image.fromarray(vis).save(out_png)

raw_dist_mean = raw_dist_sum / raw_dist_count if raw_dist_count else float('nan')
print('Pre-normalization raw dist_map mean (all samples):', raw_dist_mean)
for mode in modes:
    mode_mean = raw_dist_sum_by_mode[mode] / raw_dist_count_by_mode[mode] if raw_dist_count_by_mode[mode] else float('nan')
    print(f'Pre-normalization raw dist_map mean ({mode}):', mode_mean)

print('Done. Output in:', out_dir)

In [ ]:
# Visualize saved distance maps only; avoid writing inverted outputs a second time
dist_files = sorted(out_dir.glob('*_dist.npy'))
print(f'Found {len(dist_files)} signed distance maps in {out_dir}')

for idx in range(min(3, len(gt_files))):
    dm_npy_path = out_dir / (gt_files[idx].stem + '_dist.npy')
    dm_npy_inv_path = out_inverted_dir / (gt_files[idx].stem + '_dist_inverted.npy')
    mask_path = gt_dir / gt_files[idx].name

    dm_npy = np.load(dm_npy_path)
    dm_npy_inv = np.load(dm_npy_inv_path)
    mask = np.array(Image.open(mask_path).convert('L'))

    plt.figure(figsize=(30, 6))
    plt.subplot(1, 5, 1)
    plt.title('Original Mask')
    plt.imshow(mask, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 5, 2)
    plt.title('Distance Map')
    plt.imshow(dm_npy, cmap='magma')
    plt.axis('off')

    plt.subplot(1, 5, 3)
    plt.title('Inverted Distance Map')
    plt.imshow(dm_npy_inv, cmap='magma')
    plt.axis('off')

    plt.subplot(1, 5, 4)
    plt.title(f'DM -> Original Mask(thresholded)\n{np.sum(dm_npy > 0) / np.sum(mask > 0) * 100:.2f}% corrected')
    plt.imshow(dm_npy > 0, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 5, 5)
    plt.title(f'DM(inv) -> Original Mask(thresholded)\n{np.sum(dm_npy_inv > 0) / np.sum(mask > 0) * 100:.2f}% corrected')
    plt.imshow(dm_npy_inv > 0, cmap='gray')
    plt.axis('off')

    plt.show()
    print('distance map', np.unique(dm_npy)[::10])
    print('distance map(inverted)', np.unique(dm_npy_inv)[::10])

## MyDataset_CHASE 실제 데이터 검증 (`binary`, `dist`, `dist_inverted`)

아래 셀은 `train.py`의 CHASE split 로직(`overall_id`, `exp_id`)을 따라 실제 `data/chase`를 로드해 검증합니다.

- `dist` 요청은 `label_mode='dist'`로 실행
- `dist_inverted` 요청은 `label_mode='dist_inverted'`로 실행

참고: `MyDataset_CHASE`는 `images/`와 distance PNG를 기대하므로, 실제 폴더(`img/`, `.npy`)를 임시 호환 경로로 브리지하여 테스트합니다.

In [ ]:
def _build_chase_compat_root(real_root: Path) -> Path:
    """Create a temporary compatibility root for MyDataset_CHASE using real data only."""
    compat_root = Path(tempfile.mkdtemp(prefix='chase_real_compat_'))

    # images: MyDataset_CHASE expects images/, repo uses img/
    src_img = real_root / 'images'
    if not src_img.exists():
        raise FileNotFoundError(f'Missing source image dir: {src_img}')
    (compat_root / 'images').symlink_to(src_img.resolve(), target_is_directory=True)

    # gt can be symlinked directly
    src_gt = real_root / 'gt'
    if not src_gt.exists():
        raise FileNotFoundError(f'Missing source dir: {src_gt}')
    (compat_root / 'gt').symlink_to(src_gt.resolve(), target_is_directory=True)

    # Copy distance label directories with proper .npy handling
    # The data loader will use allow_pickle=True to load them
    src_signed = real_root / 'gt_dist'
    src_inv = real_root / 'gt_dist_inverted'
    if not src_signed.exists():
        raise FileNotFoundError(f'Missing source dir: {src_signed}')
    if not src_inv.exists():
        raise FileNotFoundError(f'Missing source dir: {src_inv}')

    out_signed = compat_root / 'gt_dist'
    out_inv = compat_root / 'gt_dist_inverted'
    out_signed.mkdir(parents=True, exist_ok=True)
    out_inv.mkdir(parents=True, exist_ok=True)

    # Copy signed distance .npy files
    signed_npy_files = sorted(src_signed.glob('*_dist.npy'))
    if len(signed_npy_files) == 0:
        raise RuntimeError('No *_dist.npy found in gt_dist')
    for npy_path in signed_npy_files:
        # Load as numpy (with allow_pickle since they may contain pickled data)
        arr = np.load(npy_path, allow_pickle=True)
        # Ensure it's a regular numpy array
        if isinstance(arr, np.ndarray):
            arr = np.asarray(arr, dtype=np.float32)
        out_path = out_signed / npy_path.name
        # Save as regular numpy array (not pickled)
        np.save(out_path, arr)
        print(f"  Copied: {npy_path.name}")

    # Copy inverted distance .npy files
    inv_npy_files = sorted(src_inv.glob('*_dist_inverted.npy'))
    if len(inv_npy_files) == 0:
        raise RuntimeError('No *_dist_inverted.npy found in gt_dist_inverted')

    for npy_path in inv_npy_files:
        # Load as numpy (with allow_pickle since they may contain pickled data)
        arr = np.load(npy_path, allow_pickle=True)
        # Ensure it's a regular numpy array
        if isinstance(arr, np.ndarray):
            arr = np.asarray(arr, dtype=np.float32)
        out_path = out_inv / npy_path.name
        # Save as regular numpy array (not pickled)
        np.save(out_path, arr)
        print(f"  Copied: {npy_path.name}")

    return compat_root

In [ ]:
# (선택) train.py 스타일로 train/test dataset 길이 확인 + 샘플 시각화
import matplotlib.pyplot as plt

args = SimpleNamespace(num_class=1, dataset='chase')
real_root = Path('data/chase')

# Distance Affinity

In [ ]:
import autorootcwd  # Auto-configure repo root and sys.path
import torch

from connect_loss import distance_affinity_matrix


def shift_like_connect(x: torch.Tensor, dy: int, dx: int) -> torch.Tensor:
    """Shift with the same convention used by connectivity_matrix_5x5/shift_8_directions."""
    # x: (B, H, W)
    b, h, w = x.shape
    out = torch.zeros_like(x)

    if dy >= 0:
        src_r0, src_r1 = 0, h - dy
        dst_r0, dst_r1 = dy, h
    else:
        src_r0, src_r1 = -dy, h
        dst_r0, dst_r1 = 0, h + dy

    if dx >= 0:
        src_c0, src_c1 = 0, w - dx
        dst_c0, dst_c1 = dx, w
    else:
        src_c0, src_c1 = -dx, w
        dst_c0, dst_c1 = 0, w + dx

    if src_r1 > src_r0 and src_c1 > src_c0:
        out[:, dst_r0:dst_r1, dst_c0:dst_c1] = x[:, src_r0:src_r1, src_c0:src_c1]

    return out


# Small deterministic sample for exact-value checks
sigma = 2.0
dist_map = torch.tensor(
    [
        [
            [0.0, 1.0, 2.0],
            [1.0, 2.0, 3.0],
            [2.0, 3.0, 4.0],
        ]
    ],
    dtype=torch.float32,
)  # (B=1, H=3, W=3)

aff = distance_affinity_matrix(dist_map, conn_num=8, sigma=sigma)
print('aff shape:', tuple(aff.shape))
assert aff.shape == (1, 8, 3, 3), f'Expected (1, 8, 3, 3), got {aff.shape}'

# Channel order in connect_loss.shift_8_directions
offsets = [
    (1, 1),   # down_right
    (1, 0),   # down
    (1, -1),  # down_left
    (0, 1),   # right
    (0, -1),  # left
    (-1, 1),  # up_right
    (-1, 0),  # up
    (-1, -1),  # up_left
]

max_diffs = []
for ch, (dy, dx) in enumerate(offsets):
    neigh = shift_like_connect(dist_map, dy=dy, dx=dx)
    expected = dist_map * neigh * torch.exp(-torch.abs(dist_map - neigh) / sigma)
    diff = (aff[:, ch] - expected).abs().max().item()
    max_diffs.append(diff)
    print(f'ch={ch}, offset=({dy:+d},{dx:+d}), max_abs_diff={diff:.3e}')

assert max(max_diffs) < 1e-6, f'Mismatch found: max diff={max(max_diffs)}'
print('✓ Per-channel computation verified against manual shifts')

# 4D input path should be equivalent to 3D input path
aff_4d = distance_affinity_matrix(dist_map.unsqueeze(1), conn_num=8, sigma=sigma)
assert torch.allclose(aff, aff_4d, atol=1e-6), '3D vs 4D input path mismatch'
print('✓ 3D/4D input shape equivalence verified')

# Sanity: larger sigma should generally reduce decay (thus larger affinity on this sample)
mean_small_sigma = distance_affinity_matrix(dist_map, conn_num=8, sigma=0.5).mean().item()
mean_big_sigma = distance_affinity_matrix(dist_map, conn_num=8, sigma=5.0).mean().item()
print(f'mean affinity (sigma=0.5): {mean_small_sigma:.6f}')
print(f'mean affinity (sigma=5.0): {mean_big_sigma:.6f}')
assert mean_big_sigma > mean_small_sigma, 'Sigma sensitivity check failed'
print('✓ Sigma decay sensitivity verified')

print('\n✅ PASS: distance_affinity_matrix all checks completed.')

In [ ]:
import math
from types import SimpleNamespace

import pandas as pd
import torch
import torch.nn.functional as F

from connect_loss import (
    Bilateral_voting,
    Bilateral_voting_kxk,
    connect_loss,
    connectivity_matrix,
    connectivity_matrix_5x5,
    distance_affinity_matrix,
)


def build_one_step_translation(size, device):
    mat = torch.zeros((size, size), dtype=torch.float32, device=device)
    for i in range(size - 1):
        mat[i, i + 1] = 1.0
    return mat


def manual_terms(loss_fn, c_map, target):
    B, _, H, W = c_map.shape
    c_map_sig = torch.sigmoid(c_map)
    class_pred = c_map_sig.view(B, loss_fn.args.num_class, loss_fn.conn_num, H, W)
    hori_t = loss_fn.hori_translation.repeat(B, 1, 1, 1)
    verti_t = loss_fn.verti_translation.repeat(B, 1, 1, 1)

    if loss_fn.conn_num == 8:
        pred, bicon_map = Bilateral_voting(class_pred, hori_t, verti_t)
    else:
        pred, bicon_map = Bilateral_voting_kxk(
            class_pred,
            hori_t,
            verti_t,
            conn_num=int(math.sqrt(loss_fn.conn_num)),
        )

    if loss_fn.label_mode == 'binary':
        if loss_fn.conn_num == 8:
            affinity_target = connectivity_matrix(target.float(), loss_fn.args.num_class)
        else:
            affinity_target = connectivity_matrix_5x5(target.float(), loss_fn.args.num_class)
        edge = loss_fn.binary_edge_target_from_affinity(affinity_target)
        vote_l = loss_fn.BCEloss(pred, target).mean()
        affinity_l = loss_fn.BCEloss(c_map_sig, affinity_target).mean()
        edge_l = loss_fn.edge_loss(bicon_map, edge)
        dice_l = loss_fn.dice_loss(pred[:, 0], target[:, 0])
        if loss_fn.args.dataset == 'chase':
            bicon_l = torch.zeros_like(vote_l)
            total = vote_l + affinity_l + edge_l + dice_l
        else:
            bicon_l = loss_fn.BCEloss(bicon_map.squeeze(1), affinity_target).mean()
            total = vote_l + affinity_l + edge_l + 0.2 * bicon_l + dice_l
        vote_target = target
    else:
        affinity_target = distance_affinity_matrix(target.float(), conn_num=loss_fn.conn_num, sigma=loss_fn.sigma)
        vote_target = loss_fn.dist_target_to_mask(target)
        edge_l, edge = loss_fn.dist_edge_loss(bicon_map, vote_target)
        vote_l = loss_fn.BCEloss(pred, vote_target).mean()
        affinity_l = loss_fn.sml1_loss(c_map_sig, affinity_target).mean()
        bicon_l = loss_fn.sml1_loss(bicon_map.view_as(affinity_target), affinity_target).mean()
        dice_l = loss_fn.dice_loss(pred[:, 0], vote_target[:, 0])
        total = vote_l + affinity_l + edge_l + 0.2 * bicon_l + dice_l

    return {
        'pred': pred,
        'bicon_map': bicon_map,
        'vote_target': vote_target,
        'affinity_target': affinity_target,
        'edge_target': edge,
        'vote': vote_l,
        'affinity': affinity_l,
        'edge': edge_l,
        'bicon': bicon_l,
        'dice': dice_l,
        'total': total,
    }


def test_single_class_forward(label_mode, conn_num=8):
    print(f"\n{'=' * 70}")
    print(f"Testing single_class_forward: label_mode={label_mode}, conn_num={conn_num}")
    print('=' * 70)

    args = SimpleNamespace(num_class=1, dataset='chase')
    device = globals().get('DEVICE', torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
    H, W = 8, 8
    hori_translation = build_one_step_translation(W, device).unsqueeze(0).unsqueeze(0)
    verti_translation = build_one_step_translation(H, device).unsqueeze(0).unsqueeze(0)

    loss_fn = connect_loss(
        args=args,
        hori_translation=hori_translation,
        verti_translation=verti_translation,
        label_mode=label_mode,
        conn_num=conn_num,
        sigma=2.0,
    )

    torch.manual_seed(0)
    c_map = torch.randn(2, 8 if conn_num == 8 else 24, H, W, device=device)
    if label_mode == 'binary':
        target = torch.randint(0, 2, (2, 1, H, W), device=device).float()
    else:
        target = torch.rand(2, 1, H, W, device=device)

    loss, loss_dict = loss_fn(c_map, target, return_details=True)
    manual = manual_terms(loss_fn, c_map, target)
    assert torch.allclose(loss, manual['total'], atol=1e-6)

    rows = []
    for key in ['vote', 'affinity', 'edge', 'bicon', 'dice', 'total']:
        rows.append({
            'term': key,
            'forward': float(loss_dict[key].detach().item()),
            'manual': float(manual[key].detach().item()),
        })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print('vote_target mean:', float(manual['vote_target'].mean().item()))
    print('affinity_target mean:', float(manual['affinity_target'].mean().item()))
    print('edge_target mean:', float(manual['edge_target'].mean().item()))
    print('pred mean:', float(manual['pred'].mean().item()))
    return float(loss.item())


print(f"\n{'#' * 70}")
print('# Current-source single_class_forward() alignment')
print('#' * 70)

results = {}
for mode in ['binary', 'dist', 'dist_inverted']:
    results[mode] = test_single_class_forward(mode, conn_num=8)
print('summary:', results)

In [ ]:
import pandas as pd
import torch
from pathlib import Path
from types import SimpleNamespace

from data_loader.GetDataset_CHASE import MyDataset_CHASE
from connect_loss import Bilateral_voting, connect_loss, connectivity_matrix, distance_affinity_matrix

RUN_DEVICE = globals().get('DEVICE', torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
print('device:', RUN_DEVICE)


def build_solver_translation(height, width, device):
    hori = torch.zeros((1, 1, width, width), dtype=torch.float32, device=device)
    for i in range(width - 1):
        hori[:, :, i, i + 1] = 1.0
    vert = torch.zeros((1, 1, height, height), dtype=torch.float32, device=device)
    for j in range(height - 1):
        vert[:, :, j, j + 1] = 1.0
    return hori, vert


real_root = Path('data/chase')
ids = ['01', '02', '03']
args = SimpleNamespace(num_class=1, dataset='chase')

binary_set = MyDataset_CHASE(args=args, train_root=str(real_root), pat_ls=ids, mode='test', label_mode='binary')
dist_set = MyDataset_CHASE(args=args, train_root=str(real_root), pat_ls=ids, mode='test', label_mode='dist')
_, binary_y0, _ = binary_set[0]
H, W = binary_y0.shape[-2:]
hori_translation, verti_translation = build_solver_translation(H, W, RUN_DEVICE)

binary_loss = connect_loss(args, hori_translation, verti_translation, label_mode='binary', conn_num=8, sigma=2.0)
dist_loss = connect_loss(args, hori_translation, verti_translation, label_mode='dist', conn_num=8, sigma=2.0)

torch.manual_seed(0)
rows = []
for idx in range(min(3, len(binary_set), len(dist_set))):
    _, binary_target, name_b = binary_set[idx]
    _, dist_target, name_d = dist_set[idx]
    assert name_b == name_d
    name = name_b

    binary_target = binary_target.unsqueeze(0).to(RUN_DEVICE)
    dist_target = dist_target.unsqueeze(0).to(RUN_DEVICE)
    logits = torch.randn(1, 8, H, W, device=RUN_DEVICE)

    binary_total, binary_terms = binary_loss(logits, binary_target, return_details=True)
    dist_total, dist_terms = dist_loss(logits, dist_target, return_details=True)

    binary_affinity = connectivity_matrix(binary_target, 1)
    dist_affinity = distance_affinity_matrix(dist_target, conn_num=dist_loss.conn_num, sigma=dist_loss.sigma)
    dist_mask = dist_loss.dist_target_to_mask(dist_target)
    binary_edge = binary_loss.binary_edge_target_from_affinity(binary_affinity)
    dist_edge = dist_loss.binary_edge_target_from_affinity(connectivity_matrix(dist_mask, 1))

    rows.extend([
        {
            'sample': name,
            'mode': 'binary',
            'raw_target_mean': float(binary_target.mean().item()),
            'vote_target_mean': float(binary_target.mean().item()),
            'affinity_target_mean': float(binary_affinity.mean().item()),
            'edge_target_mean': float(binary_edge.mean().item()),
            'total': float(binary_total.item()),
            'vote': float(binary_terms['vote'].item()),
            'affinity': float(binary_terms['affinity'].item()),
            'edge': float(binary_terms['edge'].item()),
            'bicon': float(binary_terms['bicon'].item()),
            'dice': float(binary_terms['dice'].item()),
        },
        {
            'sample': name,
            'mode': 'dist',
            'raw_target_mean': float(dist_target.mean().item()),
            'vote_target_mean': float(dist_mask.mean().item()),
            'affinity_target_mean': float(dist_affinity.mean().item()),
            'edge_target_mean': float(dist_edge.mean().item()),
            'total': float(dist_total.item()),
            'vote': float(dist_terms['vote'].item()),
            'affinity': float(dist_terms['affinity'].item()),
            'edge': float(dist_terms['edge'].item()),
            'bicon': float(dist_terms['bicon'].item()),
            'dice': float(dist_terms['dice'].item()),
        },
    ])

current_compare_df = pd.DataFrame(rows)
print(current_compare_df.to_string(index=False))
print('\nmean by mode:')
print(current_compare_df.groupby('mode').mean(numeric_only=True).to_string())

# Solver.py 현재 eval 경로 기계적 점검 (`dist`)

이 셀은 **현재 `solver.py` dist eval 경로**가 shape/runtime 측면에서 동작하는지 확인합니다.

현재 경로 핵심:
1. `pred_conn_prob = sigmoid(output)`
2. `pred_score = Bilateral_voting(pred_conn_prob)`
3. `pred_mask = dist_score_to_binary(pred_score)` with `> 0.5`
4. `gt_mask = (y_test_raw > 0)`

중요한 차이:
- 예전 `pred_score > 0`는 거의 all-foreground가 되었음
- 현재는 `pred_score > 0.5`를 사용해 그 collapse를 막음


In [ ]:
import pandas as pd
import torch
from types import SimpleNamespace

from solver import Solver
from connect_loss import Bilateral_voting

RUN_DEVICE = globals().get('DEVICE', torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
args = SimpleNamespace(num_class=1, dataset='chase', conn_num=8, resize=[960, 960], lr=1e-3, use_SDL=False, weights='', sigma=2.0)
solver = Solver(args)
hori = solver.hori_translation.to(RUN_DEVICE)
vert = solver.verti_translation.to(RUN_DEVICE)

rows = []
for logit_value in [-6, -4, -2, 0, 2, 4, 6]:
    logits = torch.full((1, 8, 960, 960), logit_value, device=RUN_DEVICE)
    pred_score, _ = Bilateral_voting(torch.sigmoid(logits).view(1, 1, 8, 960, 960), hori, vert)
    pred_mask_new = solver.dist_score_to_binary(pred_score.view(1, 1, 960, 960))
    pred_mask_old = (pred_score.view(1, 1, 960, 960) > 0).float()
    rows.append({
        'logit': logit_value,
        'pred_score_mean': float(pred_score.mean().item()),
        'legacy_fg_ratio_gt0': float(pred_mask_old.mean().item()),
        'current_fg_ratio_gt05': float(pred_mask_new.mean().item()),
    })

threshold_df = pd.DataFrame(rows)
print(threshold_df.to_string(index=False))

# Binary vs dist: current loss / input / output comparison

이 섹션은 **현재 코드 기준**으로 `binary`와 `dist`를 비교합니다.

비교 포인트:
1. raw target input 차이
2. final vote supervision target 차이
3. affinity supervision target 차이
4. evaluation output threshold 차이
5. 왜 old dist path가 collapse했는지, current path가 어떻게 그 문제를 피하는지


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from types import SimpleNamespace

from data_loader.GetDataset_CHASE import MyDataset_CHASE
from connect_loss import connectivity_matrix, distance_affinity_matrix

DIAG_DEVICE = globals().get('DEVICE', torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
args = SimpleNamespace(num_class=1, dataset='chase')
real_root = 'data/chase'
ids = ['01', '02', '03']

binary_set = MyDataset_CHASE(args=args, train_root=real_root, pat_ls=ids, mode='test', label_mode='binary')
dist_set = MyDataset_CHASE(args=args, train_root=real_root, pat_ls=ids, mode='test', label_mode='dist')

_, binary_target_raw, sample_name = binary_set[0]
_, dist_target_raw, sample_name_dist = dist_set[0]
assert sample_name == sample_name_dist

binary_target = binary_target_raw.unsqueeze(0).to(DIAG_DEVICE)
dist_target = dist_target_raw.unsqueeze(0).to(DIAG_DEVICE)
dist_mask_target = (dist_target > 0).float()

binary_affinity = connectivity_matrix(binary_target, 1)
dist_affinity = distance_affinity_matrix(dist_target, conn_num=8, sigma=2.0)

comparison_rows = [
    {
        'mode': 'binary',
        'raw_target_desc': 'binary vessel mask',
        'raw_target_mean': float(binary_target.mean().item()),
        'vote_target_desc': 'same binary mask',
        'vote_target_mean': float(binary_target.mean().item()),
        'affinity_target_desc': 'connectivity_matrix(binary mask)',
        'affinity_target_mean': float(binary_affinity.mean().item()),
    },
    {
        'mode': 'dist',
        'raw_target_desc': 'continuous distance map',
        'raw_target_mean': float(dist_target.mean().item()),
        'vote_target_desc': 'derived binary mask (y > 0)',
        'vote_target_mean': float(dist_mask_target.mean().item()),
        'affinity_target_desc': 'distance_affinity_matrix(distance map)',
        'affinity_target_mean': float(dist_affinity.mean().item()),
    },
]

input_output_compare_df = pd.DataFrame(comparison_rows)
print(input_output_compare_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(binary_target[0, 0].detach().cpu().numpy(), cmap='gray')
axes[0].set_title(f'{sample_name} binary target')
axes[0].axis('off')
axes[1].imshow(dist_target[0, 0].detach().cpu().numpy(), cmap='magma')
axes[1].set_title(f'{sample_name} dist raw target')
axes[1].axis('off')
axes[2].imshow(dist_mask_target[0, 0].detach().cpu().numpy(), cmap='gray')
axes[2].set_title(f'{sample_name} dist-derived vote target')
axes[2].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
binary_rows = []
dist_rows = []
for idx in range(min(6, len(binary_set), len(dist_set))):
    _, binary_target_raw, name_b = binary_set[idx]
    _, dist_target_raw, name_d = dist_set[idx]
    assert name_b == name_d

    binary_rows.append({
        'sample': name_b,
        'binary_target_mean': float(binary_target_raw.mean().item()),
        'binary_target_std': float(binary_target_raw.std().item()),
        'binary_pos_ratio': float((binary_target_raw > 0.5).float().mean().item()),
    })
    dist_rows.append({
        'sample': name_d,
        'dist_target_mean': float(dist_target_raw.mean().item()),
        'dist_target_std': float(dist_target_raw.std().item()),
        'dist_mask_ratio': float((dist_target_raw > 0).float().mean().item()),
    })

binary_stats_df = pd.DataFrame(binary_rows)
dist_stats_df = pd.DataFrame(dist_rows)
merged_stats_df = binary_stats_df.merge(dist_stats_df, on='sample')
print(merged_stats_df.to_string(index=False))

In [ ]:
from connect_loss import connect_loss


def build_solver_translation(height, width, device):
    hori = torch.zeros((1, 1, width, width), dtype=torch.float32, device=device)
    for i in range(width - 1):
        hori[:, :, i, i + 1] = 1.0
    vert = torch.zeros((1, 1, height, height), dtype=torch.float32, device=device)
    for j in range(height - 1):
        vert[:, :, j, j + 1] = 1.0
    return hori, vert


H, W = binary_target.shape[-2:]
hori_translation, verti_translation = build_solver_translation(H, W, DIAG_DEVICE)
binary_loss = connect_loss(args, hori_translation, verti_translation, label_mode='binary', conn_num=8, sigma=2.0)
dist_loss = connect_loss(args, hori_translation, verti_translation, label_mode='dist', conn_num=8, sigma=2.0)

torch.manual_seed(0)
loss_rows = []
for idx in range(min(6, len(binary_set), len(dist_set))):
    _, binary_target_raw, name_b = binary_set[idx]
    _, dist_target_raw, name_d = dist_set[idx]
    assert name_b == name_d
    logits = torch.randn(1, 8, H, W, device=DIAG_DEVICE)

    binary_total, binary_terms = binary_loss(logits, binary_target_raw.unsqueeze(0).to(DIAG_DEVICE), return_details=True)
    dist_total, dist_terms = dist_loss(logits, dist_target_raw.unsqueeze(0).to(DIAG_DEVICE), return_details=True)

    for mode, total, terms in [
        ('binary', binary_total, binary_terms),
        ('dist', dist_total, dist_terms),
    ]:
        loss_rows.append({
            'sample': name_b,
            'mode': mode,
            'total': float(total.item()),
            'vote': float(terms['vote'].item()),
            'affinity': float(terms['affinity'].item()),
            'edge': float(terms['edge'].item()),
            'bicon': float(terms['bicon'].item()),
            'dice': float(terms['dice'].item()),
        })

current_loss_compare_df = pd.DataFrame(loss_rows)
print(current_loss_compare_df.to_string(index=False))
print('\nmean by mode:')
print(current_loss_compare_df.groupby('mode').mean(numeric_only=True).to_string())

In [ ]:
from connect_loss import connect_loss

H, W = binary_target.shape[-2:]
hori_translation, verti_translation = build_solver_translation(H, W, DIAG_DEVICE)
binary_loss = connect_loss(args, hori_translation, verti_translation, label_mode='binary', conn_num=8, sigma=2.0)
dist_loss = connect_loss(args, hori_translation, verti_translation, label_mode='dist', conn_num=8, sigma=2.0)

binary_sample = binary_set[0][1].unsqueeze(0).to(DIAG_DEVICE)
dist_sample = dist_set[0][1].unsqueeze(0).to(DIAG_DEVICE)

rows = []
for name, logit_value in [('zero_logits', -20.0), ('mid_logits', 0.0), ('high_logits', 20.0)]:
    logits = torch.full((1, 8, H, W), logit_value, device=DIAG_DEVICE)
    _, binary_terms = binary_loss(logits, binary_sample, return_details=True)
    _, dist_terms = dist_loss(logits, dist_sample, return_details=True)
    rows.append({
        'case': name,
        'mode': 'binary',
        'total': float(binary_terms['total'].item()),
        'vote': float(binary_terms['vote'].item()),
        'edge': float(binary_terms['edge'].item()),
        'dice': float(binary_terms['dice'].item()),
    })
    rows.append({
        'case': name,
        'mode': 'dist',
        'total': float(dist_terms['total'].item()),
        'vote': float(dist_terms['vote'].item()),
        'edge': float(dist_terms['edge'].item()),
        'dice': float(dist_terms['dice'].item()),
    })

logit_case_df = pd.DataFrame(rows)
print(logit_case_df.to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd

candidate_runs = [
    Path('output/dist_lossfix_smoke30/dist_8/results_1.csv'),
    Path('output/dist_lossfix_smoke3/dist_8/results_1.csv'),
]
existing = [p for p in candidate_runs if p.exists()]
if not existing:
    print('No dist loss-fix smoke results found yet.')
else:
    for path in existing:
        print(f'\nResults from {path}:')
        df = pd.read_csv(path)
        print(df.to_string(index=False))

## Current Conclusion

현재 코드 기준 `binary`와 `dist`의 핵심 차이는 다음과 같습니다.

- `binary`
  - raw target input: binary vessel mask
  - final vote supervision target: binary vessel mask
  - affinity target: `connectivity_matrix(binary mask)`
  - eval output: voted score -> binary mask
- `dist`
  - raw target input: continuous distance map
  - final vote supervision target: **distance map 자체가 아니라** `y > 0`으로 만든 binary vessel mask
  - affinity target: `distance_affinity_matrix(distance map)`
  - eval output: voted score -> `> 0.5` thresholded binary mask

왜 old dist path가 실패했는가:

- final voted output을 raw distance map에 직접 regression 하면서
- eval에서는 `pred_score > 0`를 사용했기 때문에
- training에서는 near-zero trivial solution이 너무 유리했고
- eval에서는 non-negative score map이 거의 all-foreground가 되었습니다.

현재 해결 방식:

- final segmentation output은 `binary`와 동일하게 binary mask로 직접 supervision
- distance map 정보는 affinity / bicon auxiliary supervision으로 유지
- eval threshold는 `> 0.5`로 변경

실무적 의미:

- `dist`는 이제 최종 segmentation 관점에서는 `binary`와 같은 목표를 학습합니다.
- 차이는 final output이 아니라 **auxiliary target**에 있습니다.
- 즉, `dist`의 역할은 “distance prior를 directional affinity 학습에 추가”하는 쪽으로 정리됩니다.
